El siguiente código es una visita al tutorial https://keras.io/examples/nlp/neural_machine_translation_with_transformer/ una implementación del modelo transformer empleando Keras con el fin de realizar traducciones Ingles $\rightarrow$ Español. 

La documentación del modélo Transformer se realiza en función de los videos del canal 3Blue1Brown https://www.youtube.com/watch?v=wjZofJX0v4M y https://www.youtube.com/watch?v=eMlx5fFNoYc. Ambos son videos que resumen los conceptos expuestos por el artículo "Attetion is all you need" de Vaswani et al. 2017.

### Representación del texto: Tokenización y Embedding

Antes de que un modelo Transformer pueda procesar texto, es necesario transformar las palabras en objetos numéricos sobre los cuales se puedan aplicar operaciones matriciales. El primer paso consiste en dividir el texto de entrada en unidades más pequeñas llamadas *tokens* (que pueden ser palabras completas o fragmentos de palabras) y asignarle a cada uno un índice dentro de un vocabulario de tamaño $V$.

Sin embargo, trabajar únicamente con índices enteros no captura ninguna relación entre palabras. Para resolver esto se introduce la etapa de **Embedding**, en la cual a cada token se le asocia un vector en un espacio de alta dimensión $\mathbb{R}^{d}$ mediante una matriz entrenable $W_E \in \mathbb{R}^{V \times d}$. Así pues, una secuencia de $N$ tokens queda representada como una colección de vectores:

$$ \text{token}_i \xrightarrow{\text{embedding}} \vec{E}_i \in \mathbb{R}^{d}, \qquad i = 1, 2, \dots, N $$

Lo interesante de esta representación es que, durante el entrenamiento, el espacio vectorial adquiere una estructura geométrica donde las direcciones codifican significado semántico. En otras palabras, palabras con significados cercanos apuntan en direcciones similares y las relaciones conceptuales pueden interpretarse como desplazamientos vectoriales, por ejemplo:

$$ \vec{E}(\text{reina}) \approx \vec{E}(\text{rey}) - \vec{E}(\text{hombre}) + \vec{E}(\text{mujer}) $$

Además, como el Transformer procesa todos los tokens en paralelo y no de forma secuencial, el embedding por sí solo no distingue en qué lugar de la oración aparece cada palabra. Por tanto, a cada vector $\vec{E}_i$ se le suma un vector de **codificación posicional** $\vec{P}_i \in \mathbb{R}^{d}$ que codifica la posición $i$ dentro de la secuencia:

$$ \vec{E}_i \leftarrow \vec{E}_i + \vec{P}_i $$

Note que hasta este punto cada vector $\vec{E}_i$ contiene únicamente el significado aislado de su token y su posición, sin incorporar todavía el contexto del resto de la oración (por ejemplo, cómo un adjetivo modifica a un sustantivo o cuál es el sentido correcto de una palabra ambigua según las palabras vecinas). Precisamente para permitir que estos vectores intercambien información entre sí surge el mecanismo de atención.

### Mecanismo de Atención (Self-Attention)

El objetivo de una capa de atención es actualizar cada vector de embedding $\vec{E}_i$ de modo que incorpore información contextual proveniente de los demás tokens de la secuencia. Es decir, buscamos calcular un desplazamiento $\Delta \vec{E}_i$ tal que el nuevo vector enriquecido por el contexto sea:

$$ \vec{E}_i' = \vec{E}_i + \Delta \vec{E}_i $$

Para decidir qué palabras deben influir sobre cuáles y de qué manera, el mecanismo de atención introduce tres matrices de pesos entrenables: **Query** ($W_Q$), **Key** ($W_K$) y **Value** ($W_V$).

En primer lugar, cada vector $\vec{E}_i$ se proyecta mediante las matrices $W_Q, W_K \in \mathbb{R}^{d_k \times d}$ hacia un espacio de dimensión menor $d_k$:

$$ \vec{Q}_i = W_Q \vec{E}_i \qquad , \qquad \vec{K}_i = W_K \vec{E}_i $$

Intuitivamente, el vector consulta $\vec{Q}_i$ (*Query*) representa una "pregunta" que el token $i$ hace al resto de la oración sobre qué tipo de contexto está buscando (por ejemplo, un sustantivo buscando adjetivos que lo describan). Por su parte, el vector clave $\vec{K}_j$ (*Key*) representa la "etiqueta" o respuesta que ofrece el token $j$ para indicar qué tipo de información puede aportar.

Para medir qué tan bien responde el token $j$ a la pregunta del token $i$, calculamos el producto punto $\vec{Q}_i \cdot \vec{K}_j$. Si ambos vectores apuntan en direcciones similares, el producto punto será grande y positivo, lo cual indica que el token $i$ debe "prestar atención" al token $j$; si el valor es pequeño o negativo, ambos tokens tienen poca relación.

Como queremos expresar estas relevancias como pesos normalizados entre $0$ y $1$ que sumen $1$, dividimos el producto punto entre $\sqrt{d_k}$ (para evitar valores numéricos demasiado grandes que saturen la exponencial) y aplicamos la función $\text{softmax}$ sobre todos los tokens $j$:

$$ \alpha_{ij} = \text{softmax}_j \left( \frac{\vec{Q}_i \cdot \vec{K}_j}{\sqrt{d_k}} \right) = \frac{\exp\left(\frac{\vec{Q}_i \cdot \vec{K}_j}{\sqrt{d_k}}\right)}{\sum_{m=1}^{N} \exp\left(\frac{\vec{Q}_i \cdot \vec{K}_m}{\sqrt{d_k}}\right)} $$

donde $\alpha_{ij}$ indica la fracción de atención que el token $i$ le asigna al token $j$. Note que en tareas donde un token no debe "ver" palabras futuras (como ocurre en el decodificador al generar una traducción palabra por palabra), se aplica un enmascaramiento (*masking*) asignando $-\infty$ a las posiciones futuras antes de la $\text{softmax}$, logrando así que sus pesos sean exactamente $0$.

Una vez sabemos *cuánto* importa cada token $j$, necesitamos definir *qué información* se va a transferir para modificar a $\vec{E}_i$. Para ello multiplicamos cada embedding por la matriz de valores $W_V \in \mathbb{R}^{d \times d}$:

$$ \vec{V}_j = W_V \vec{E}_j $$

El vector $\vec{V}_j$ (*Value*) representa el cambio que el token $j$ aportaría al significado de otro token si este le prestara toda su atención. Así pues, la actualización total $\Delta \vec{E}_i$ se obtiene como la suma ponderada de todos los vectores de valor según los pesos de atención:

$$ \Delta \vec{E}_i = \sum_{j=1}^{N} \alpha_{ij} \vec{V}_j $$

y finalmente sumamos esta contribución al embedding original para obtener $\vec{E}_i' = \vec{E}_i + \Delta \vec{E}_i$.

### Forma matricial y Atención Multicabeza (Multi-Head Attention)

En la práctica, en lugar de calcular estas operaciones token por token, todos los vectores de la secuencia de longitud $N$ se apilan como filas en matrices $Q, K \in \mathbb{R}^{N \times d_k}$ y $V \in \mathbb{R}^{N \times d_v}$. De esta forma, todos los productos punto entre consultas y claves se obtienen simultáneamente mediante el producto matricial $Q K^T \in \mathbb{R}^{N \times N}$, y el mecanismo completo de atención se resume en la expresión matricial propuesta por Vaswani et al.:

$$ \text{Attention}(Q, K, V) = \text{softmax}\left( \frac{Q K^T}{\sqrt{d_k}} \right) V $$

donde la función $\text{softmax}$ se aplica por filas sobre la matriz de puntajes $\frac{Q K^T}{\sqrt{d_k}}$, produciendo la matriz de pesos de atención que luego multiplica a $V$.

Ahora bien, en una oración existen múltiples tipos de relaciones contextuales al mismo tiempo (adjetivos que modifican sustantivos, pronombres que refieren a sujetos, relaciones entre verbos y objetos, entre otras). Una sola terna de matrices $(W_Q, W_K, W_V)$ únicamente captura un patrón de atención a la vez. Para superar esto se emplea la **Atención Multicabeza** (*Multi-Head Attention*), donde se ejecutan $h$ cabezas de atención en paralelo, cada una con sus propias matrices entrenables $(W_Q^{(l)}, W_K^{(l)}, W_V^{(l)})$ para $l = 1, \dots, h$. Los resultados de todas las cabezas se combinan (concatenándolos y proyectándolos de vuelta a la dimensión $d$ mediante una matriz de salida $W_O$) para construir la actualización final del embedding.

Por último, como en este laboratorio abordamos un problema de traducción Inglés $\rightarrow$ Español, el modelo utiliza una arquitectura **Encoder-Decoder**. Recordemos que además de la auto-atención (*self-attention*) donde $Q$, $K$ y $V$ provienen de la misma secuencia, el decodificador incorpora una capa de **atención cruzada** (*cross-attention*): en esta, las consultas $Q$ provienen de la oración que se está generando en español, mientras que las claves $K$ y los valores $V$ provienen de la salida del codificador en inglés. De este modo, al generar cada palabra en español, el modelo puede enfocar su atención sobre las palabras relevantes de la frase original en inglés. Estas capas de atención, alternadas con capas densas (MLP) y normalización, conforman los bloques `TransformerEncoder` y `TransformerDecoder` que implementaremos a continuación.

In [1]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import pathlib
import random
import string
import re
import numpy as np

import tensorflow.data as tf_data
import tensorflow.strings as tf_strings

import keras
from keras import layers
from keras import ops
from keras.layers import TextVectorization

I0000 00:00:1793895792.081925  724888 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793895792.118744  724888 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793895793.446754  724888 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Downloading the data

In [2]:
text_file = keras.utils.get_file(
    fname="spa-eng.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip",
    extract=True,
)
text_file = pathlib.Path(text_file).parent / "spa-eng_extracted" /"spa-eng" / "spa.txt"


In [3]:
print(text_file)

/home/daniel/.keras/datasets/spa-eng_extracted/spa-eng/spa.txt


### Parsing the data

In [4]:
with open(text_file) as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    eng, spa = line.split("\t")
    spa = "[start] " + spa + " [end]"
    text_pairs.append((eng, spa))

In [5]:
for _ in range(5):
    print(random.choice(text_pairs))


('I will be free tomorrow afternoon.', '[start] Mañana en la tarde estoy libre. [end]')
('What do you want then?', '[start] ¿Qué quieres entonces? [end]')
("It's a town of 3,000 people.", '[start] Es un pueblo de tres mil almas. [end]')
('I can take care of myself, you know.', '[start] Puedo cuidar de mí mismo, ¿sabes? [end]')
('Your fly is open!', '[start] ¡Tienes el cierre abierto! [end]')


In [6]:
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples :]

print(f"{len(text_pairs)} total pairs")
print(f"{len(train_pairs)} training pairs")
print(f"{len(val_pairs)} validation pairs")
print(f"{len(test_pairs)} test pairs")


118964 total pairs
83276 training pairs
17844 validation pairs
17844 test pairs


### Vectorizing the text data

Este paso de aca corresponde al Embedding, en este caso se usara un método nativo

In [7]:
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

vocab_size = 15000
sequence_length = 20
batch_size = 64


def custom_standardization(input_string):
    lowercase = tf_strings.lower(input_string)
    return tf_strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")


eng_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
spa_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_eng_texts = [pair[0] for pair in train_pairs]
train_spa_texts = [pair[1] for pair in train_pairs]
eng_vectorization.adapt(train_eng_texts)
spa_vectorization.adapt(train_spa_texts)


E0000 00:00:1793895794.504227  724888 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
def format_dataset(eng, spa):
    eng = eng_vectorization(eng)
    spa = spa_vectorization(spa)
    return (
        {
            "encoder_inputs": eng,
            "decoder_inputs": spa[:, :-1],
        },
        spa[:, 1:],
    )


def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf_data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset)
    return dataset.cache().shuffle(2048).prefetch(16)


train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)


In [9]:
for inputs, targets in train_ds.take(1):
    print(f'inputs["encoder_inputs"].shape: {inputs["encoder_inputs"].shape}')
    print(f'inputs["decoder_inputs"].shape: {inputs["decoder_inputs"].shape}')
    print(f"targets.shape: {targets.shape}")


inputs["encoder_inputs"].shape: (64, 20)
inputs["decoder_inputs"].shape: (64, 20)
targets.shape: (64, 20)


### Building the model

In [10]:
import keras.ops as ops


class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, attention_mask=padding_mask
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config


class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config


class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(latent_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        inputs, encoder_outputs = inputs
        causal_mask = self.get_causal_attention_mask(inputs)

        if mask is None:
            inputs_padding_mask, encoder_outputs_padding_mask = None, None
        else:
            inputs_padding_mask, encoder_outputs_padding_mask = mask

        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=causal_mask,
            query_mask=inputs_padding_mask,
        )
        out_1 = self.layernorm_1(inputs + attention_output_1)

        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            query_mask=inputs_padding_mask,
            key_mask=encoder_outputs_padding_mask,
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)

    def get_causal_attention_mask(self, inputs):
        input_shape = ops.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = ops.arange(sequence_length)[:, None]
        j = ops.arange(sequence_length)
        mask = ops.cast(i >= j, dtype="int32")
        mask = ops.reshape(mask, (1, input_shape[1], input_shape[1]))
        return ops.tile(mask, [batch_size, 1, 1])

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "latent_dim": self.latent_dim,
                "num_heads": self.num_heads,
            }
        )
        return config


We assemble the end-to-end model

In [ ]:
embed_dim = 256
latent_dim = 2048
num_heads = 8

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, latent_dim, num_heads)(x)
encoder = keras.Model(encoder_inputs, encoder_outputs)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)([x, encoder_outputs])
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)

transformer = keras.Model(
    {"encoder_inputs": encoder_inputs, "decoder_inputs": decoder_inputs},
    decoder_outputs,
    name="transformer",
)


### Training our model

In [12]:
epochs = 1  # This should be at least 30 for convergence

transformer.summary()
transformer.compile(
    "rmsprop",
    loss=keras.losses.SparseCategoricalCrossentropy(ignore_class=0),
    metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=epochs, validation_data=val_ds)


Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  3,845,120 │ encoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  3,845,120 │ decoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, None, 256) │  3,155,456 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decoder │ (None, None, 256) │  5,259,520 │ positional_embed… │
│ (TransformerDecode… │                   │            │ transformer_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, None, 256) │          0 │ transformer_deco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, None,      │  3,855,000 │ dropout_3[0][0]   │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,960,216 (76.14 MB)

 Trainable params: 19,960,216 (76.14 MB)

 Non-trainable params: 0 (0.00 B)

/home/daniel/Workspace/venv/universidad/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1302/1302 ━━━━━━━━━━━━━━━━━━━━ 602s 460ms/step - accuracy: 0.3654 - loss: 4.0295 - val_accuracy: 0.4898 - val_loss: 2.7854


In [14]:
spa_vocab = spa_vectorization.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
max_decoded_sentence_length = 20


def decode_sequence(input_sentence):
    tokenized_input_sentence = eng_vectorization([input_sentence])
    decoded_sentence = "[start]"
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = spa_vectorization([decoded_sentence])[:, :-1]
        predictions = transformer(
            {
                "encoder_inputs": tokenized_input_sentence,
                "decoder_inputs": tokenized_target_sentence,
            }
        )

        # ops.argmax(predictions[0, i, :]) is not a concrete value for jax here
        sampled_token_index = ops.convert_to_numpy(
            ops.argmax(predictions[0, i, :])
        ).item(0)
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break
    return decoded_sentence


test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(30):
    input_sentence = random.choice(test_eng_texts)
    translated = decode_sequence(input_sentence)


In [16]:

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(30):
    input_sentence = random.choice(test_eng_texts)
    translated = decode_sequence(input_sentence)

    print(input_sentence)
    print(translated)
    print()

I thought a bunch of people would go water skiing with us, but absolutely no one else showed up.
[start] pensé que nos [UNK] de personas [UNK] a la gente pero no nos [UNK] con una gente pero no nos

Don't sneak out of the concert!
[start] no se [UNK] el concierto [end]

Stop stalling and do what I told you.
[start] deja de lo que te lo que te dije [end]

You're easygoing.
[start] estás [UNK] [end]

I wouldn't want it even if they were giving it away.
[start] no quería que se lo que ellos se lo [UNK] [end]

A monument was erected in memory of the deceased.
[start] un [UNK] estaba en la carrera [end]

The water in the lake is cool.
[start] el agua está en el lago [end]

I want to do a good job.
[start] quiero hacer un buen trabajo [end]

Why is Tom still here?
[start] por qué tom está aquí [end]

Did you see him go out?
[start] lo has visto [end]

He was in prison for life.
[start] Él estaba en la vida [end]

Now I understand.
[start] ahora lo entiendo [end]

Can you hear that?
[start] p